### Configure PyTorch CUDA memory allocation settings to prevent fragmentation

In [ ]:
 import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

### Check GPU availability and display GPU information

In [ ]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if gpu_info.find("failed") >= 0:
    print("Not connected to a GPU")
else:
    print(gpu_info)

Mon Apr  6 02:13:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Install required Python packages (datasets, transformers, torchaudio, jiwer, accelerate, evaluate)

In [ ]:
 %%capture
!pip install datasets==3.6.0
!pip install transformers==4.57.1
!pip install torchaudio
!pip install jiwer
!pip install accelerate -U
!pip install evaluate

### Install system dependencies and clone the KenLM language model toolkit

In [ ]:
!sudo apt-get update
!sudo apt-get install build-essential libboost-all-dev cmake zlib1g-dev libbz2-dev liblzma-dev
!sudo apt-get install libboost-all-dev libeigen3-dev
!git clone https://github.com/kpu/kenlm
%cd kenlm
!pip install e .

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,482 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,98

### Build and install KenLM from source

In [ ]:
!mkdir build
%cd build
!cmake ..
!make -j 4
!sudo make install

mkdir: cannot create directory ‘build’: File exists
/content/kenlm/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMake Warning (dev) at CMakeLists.txt:101 (find_package):
  Policy CMP0167 is not set: The FindBoost module is removed.  Run "cmake
  --help-policy CMP0167" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.

-- Found Boost: /usr/lib/x86_64-linux-gnu/cmake/Boost-1.74.0/BoostConfig.cmake (found suitable v

### Add KenLM binaries to the system PATH and verify installation

In [ ]:
import os
os.environ['PATH'] += ":/content/kenlm/build/bin"  # Use your exact path
!echo $PATH  # Verify
!which lmplz  # Test

/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/content/kenlm/build/bin
/usr/local/bin/lmplz


### Import all required libraries for data processing, model training, and evaluation

In [ ]:
import re
import json
import unicodedata
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union

from datasets import load_dataset, Audio, concatenate_datasets
import evaluate

from transformers import (
    Wav2Vec2CTCTokenizer,
    SeamlessM4TFeatureExtractor,
    Wav2Vec2BertProcessor,
    Wav2Vec2BertForCTC,
    TrainingArguments,
    Trainer,
)

### Load Dhivehi Common Voice dataset (train+validation and test) and remove unused metadata columns

In [ ]:
dv_train = load_dataset("fsicoli/common_voice_22_0", "dv", split="train+validation", trust_remote_code=True)
dv_test  = load_dataset("fsicoli/common_voice_22_0", "dv", split="test", trust_remote_code=True)

dv_train = dv_train.remove_columns(["accent","age","client_id","down_votes","gender","locale","segment","up_votes"])
dv_test  = dv_test.remove_columns(["accent","age","client_id","down_votes","gender","locale","segment","up_votes"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

common_voice_22_0.py: 0.00B [00:00, ?B/s]

languages.py: 0.00B [00:00, ?B/s]

release_stats.py: 0.00B [00:00, ?B/s]

...


Reading metadata...: 2222it [00:00, 120993.20it/s]


Generating other split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 15104it [00:00, 122756.63it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 1652it [00:00, 112675.67it/s]


### Load Turkish Common Voice dataset (train+validation and test) and remove unused metadata columns

In [ ]:
tr_train = load_dataset(
    "fsicoli/common_voice_22_0",
    "tr",
    split="train+validation",
    trust_remote_code=True
)

tr_test = load_dataset(
    "fsicoli/common_voice_22_0",
    "tr",
    split="test",
    trust_remote_code=True
)

tr_train = tr_train.remove_columns([
    "accent","age","client_id","down_votes","gender","locale","segment","up_votes"
])

tr_test = tr_test.remove_columns([
    "accent","age","client_id","down_votes","gender","locale","segment","up_votes"
])

audio/tr/train/tr_train_0.tar:   0%|          | 0.00/964M [00:00<?, ?B/s]

audio/tr/train/tr_train_1.tar:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

audio/tr/dev/tr_dev_0.tar:   0%|          | 0.00/279M [00:00<?, ?B/s]

audio/tr/test/tr_test_0.tar:   0%|          | 0.00/343M [00:00<?, ?B/s]

audio/tr/other/tr_other_0.tar:   0%|          | 0.00/4.21M [00:00<?, ?B/s]

...


Reading metadata...: 11784it [00:00, 141793.93it/s]


Generating other split: 0 examples [00:00, ? examples/s]


Reading metadata...: 116it [00:00, 80847.34it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 4893it [00:00, 136247.77it/s]


### Define text cleaning functions for Dhivehi (remove punctuation, Arabic, Latin) and Turkish (normalize and filter non-Turkish characters)

In [ ]:
import re
import unicodedata

# -------- Dhivehi cleaning --------
chars_to_remove_regex_dv = r'[\,\?\.\!\-\;\:\"\“\%\‘\”\�\'\»\«]'
arabic_pattern = re.compile(r"[\u0600-\u06FF]")

def dv_clean(batch):
    s = re.sub(chars_to_remove_regex_dv, "", batch["sentence"]).lower()

    # remove Arabic rows
    if arabic_pattern.search(s):
        batch["sentence"] = ""
        return batch

    # remove latin
    s = re.sub(r"[a-z]+", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    batch["sentence"] = s
    return batch


# -------- Turkish cleaning --------
chars_to_remove_regex_tr = r'[\,\?\.\!\-\;\:\"\“\%\‘\”\�\(\)\'\”\‘‘\’\ʻ\”\“\–\—\…\»\«]'
turkish_allowed = re.compile(r"^[a-zçğıöşü\s]+$")

def tr_clean(batch):
    s = unicodedata.normalize("NFC", batch["sentence"]).lower()
    s = re.sub(chars_to_remove_regex_tr, "", s)
    s = re.sub(r"[0-9]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    batch["sentence"] = s
    return batch

### Apply cleaning functions and filter out empty sentences from both Dhivehi and Turkish datasets

In [ ]:
dv_train = dv_train.map(dv_clean).filter(lambda x: len(x["sentence"]) > 0)
dv_test  = dv_test.map(dv_clean).filter(lambda x: len(x["sentence"]) > 0)

tr_train = tr_train.map(tr_clean).filter(lambda x: bool(turkish_allowed.match(x["sentence"])) if x["sentence"] else False)
tr_test  = tr_test.map(tr_clean).filter(lambda x: bool(turkish_allowed.match(x["sentence"])) if x["sentence"] else False)

Map:   0%|          | 0/4897 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4897 [00:00<?, ? examples/s]

Map:   0%|          | 0/2222 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2222 [00:00<?, ? examples/s]

Map:   0%|          | 0/52160 [00:00<?, ? examples/s]

Filter:   0%|          | 0/52160 [00:00<?, ? examples/s]

Map:   0%|          | 0/11784 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11784 [00:00<?, ? examples/s]

### Resample all audio to 16kHz for both Dhivehi and Turkish datasets

In [ ]:
dv_train = dv_train.cast_column("audio", Audio(sampling_rate=16_000))
dv_test  = dv_test.cast_column("audio", Audio(sampling_rate=16_000))
tr_train = tr_train.cast_column("audio", Audio(sampling_rate=16_000))
tr_test  = tr_test.cast_column("audio", Audio(sampling_rate=16_000))

### Extract the combined character vocabulary from both Dhivehi and Turkish datasets

In [ ]:
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

all_train_text = concatenate_datasets([
    dv_train.select_columns(["sentence"]),
    tr_train.select_columns(["sentence"]),
])
all_test_text = concatenate_datasets([
    dv_test.select_columns(["sentence"]),
    tr_test.select_columns(["sentence"]),
])

vocab_train = all_train_text.map(
    extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True,
    remove_columns=all_train_text.column_names
)
vocab_test = all_test_text.map(
    extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True,
    remove_columns=all_test_text.column_names
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}

# Space -> word delimiter |
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]


# Specials
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)

print("Vocab size:", len(vocab_dict))

Map:   0%|          | 0/53689 [00:00<?, ? examples/s]

Map:   0%|          | 0/13147 [00:00<?, ? examples/s]

Vocab size: 87


### Initialize the CTC tokenizer, feature extractor, and processor

In [ ]:
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = SeamlessM4TFeatureExtractor(
    feature_size=80,
    num_mel_bins=80,
    sampling_rate=16000,
    padding_value=0.0,
)

processor = Wav2Vec2BertProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

### Write all transcriptions to a text file for KenLM language model training

In [ ]:
with open("lm_corpus.txt", "w", encoding="utf-8") as f:
    for ex in all_train_text:
        s = ex["sentence"].strip()
        if s:
            f.write(s + "\n")
    for ex in all_test_text:
        s = ex["sentence"].strip()
        if s:
            f.write(s + "\n")

print("Wrote LM corpus to lm_corpus.txt")

Wrote LM corpus to lm_corpus.txt


### Define the dataset preparation function: extract audio features and encode text labels, then apply to train and test sets

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["input_length"] = len(batch["input_features"])
    batch["labels"] = processor(text=batch["sentence"]).input_ids
    return batch

dv_train_p = dv_train.map(prepare_dataset, remove_columns=dv_train.column_names)
dv_test_p  = dv_test.map(prepare_dataset,  remove_columns=dv_test.column_names)
si_train_p = tr_train.map(prepare_dataset, remove_columns=tr_train.column_names)
si_test_p  = tr_test.map(prepare_dataset,  remove_columns=tr_test.column_names)

Map:   0%|          | 0/4639 [00:00<?, ? examples/s]

Map:   0%|          | 0/2107 [00:00<?, ? examples/s]

Map:   0%|          | 0/49050 [00:00<?, ? examples/s]

Map:   0%|          | 0/11040 [00:00<?, ? examples/s]

### Concatenate Dhivehi and Turkish processed datasets into unified train and test sets

In [ ]:
train = concatenate_datasets([dv_train_p, si_train_p]).shuffle(seed=42)
test  = concatenate_datasets([dv_test_p,  si_test_p]).shuffle(seed=42)

print("Train size:", len(train))
print("Test size:", len(test))


Train size: 53689
Test size: 13147


### Define a custom data collator that pads input features and labels for CTC training

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2BertProcessor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

### Train a trigram KenLM language model from the corpus

In [ ]:
!lmplz -o 3 < lm_corpus.txt > lm.arpa

=== 1/5 Counting and sorting n-grams ===
Reading /content/kenlm/build/lm_corpus.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 343727 types 72580
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:870960 2:24939995136 3:46762491904
Statistics:
1 72580 D1=0.690644 D2=1.09753 D3+=1.46309
2 251422 D1=0.873067 D2=1.17106 D3+=1.36289
3 307404 D1=0.929733 D2=1.34126 D3+=1.60236
Memory estimate for binary LM:
type       kB
probing 13139 assuming -p 1.5
probing 14895 assuming -r models -p 1.5
trie     6540 without quantization
trie     4238 assuming -q 8 -b 8 quantization 
trie     6179 assuming -a 22 array pointer compression
trie     3876 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:870960 2:4022752 3:6148080
-

### Install pyctcdecode library for CTC beam search decoding with language model support

In [ ]:
!pip install pyctcdecode

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.6/529.6 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 130.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy

### Build a CTC beam search decoder with KenLM language model integration

In [ ]:
from pyctcdecode import build_ctcdecoder

# get labels in ID order
vocab = processor.tokenizer.get_vocab()  # token -> id
id2token = {v: k for k, v in vocab.items()}
labels = [id2token[i] for i in range(len(id2token))]

# path to your trained KenLM model (arpa or binary)
kenlm_model_path = "lm.arpa"  # <-- change if you use another path/name

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=kenlm_model_path,
    alpha=0.5,  # tune on dev
    beta=1.0  # tune on dev
)


### Define evaluation metrics function computing WER and CER using greedy decoding

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    label_ids = pred.label_ids.copy()

    pred_ids = np.argmax(pred_logits, axis=-1)
    pred_str = processor.batch_decode(pred_ids)

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

### Load the pre-trained W2V-BERT 2.0 model with an adapter and CTC head for fine-tuning

In [ ]:
model = Wav2Vec2BertForCTC.from_pretrained(
    "facebook/w2v-bert-2.0",
    add_adapter=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

Some weights of Wav2Vec2BertForCTC were not initialized from the model checkpoint at facebook/w2v-bert-2.0 and are newly initialized: ['lm_head.bias', 'lm_head.weight', 'wav2vec2_bert.adapter.layers.0.ffn.intermediate_dense.bias', 'wav2vec2_bert.adapter.layers.0.ffn.intermediate_dense.weight', 'wav2vec2_bert.adapter.layers.0.ffn.output_dense.bias', 'wav2vec2_bert.adapter.layers.0.ffn.output_dense.weight', 'wav2vec2_bert.adapter.layers.0.ffn_layer_norm.bias', 'wav2vec2_bert.adapter.layers.0.ffn_layer_norm.weight', 'wav2vec2_bert.adapter.layers.0.residual_conv.bias', 'wav2vec2_bert.adapter.layers.0.residual_conv.weight', 'wav2vec2_bert.adapter.layers.0.residual_layer_norm.bias', 'wav2vec2_bert.adapter.layers.0.residual_layer_norm.weight', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_k.bias', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_k.weight', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_out.bias', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_out.weight', 'wav2vec2_ber

### Configure training hyperparameters: batch size, learning rate, logging, and checkpointing (no evaluation during training)

In [ ]:
from transformers import TrainingArguments, Trainer, Wav2Vec2BertForCTC

training_args = TrainingArguments(
    output_dir="./outputs",
    group_by_length=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    gradient_checkpointing=True,
    fp16=True,

    eval_strategy="no",   # no evaluation
    logging_strategy="steps",
    logging_steps=300,

    save_strategy="steps",
    save_steps=2700,

    learning_rate=5e-5,
    warmup_steps=500,
    save_total_limit=2,

    load_best_model_at_end=False,   # must be False if no evaluation
    push_to_hub=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    train_dataset=train,
    tokenizer=processor,
)

trainer.train()

/tmp/ipykernel_2468/1702521174.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
300,2348.839200
600,756.175100
900,576.865200
1200,462.353900
1500,334.874300
1800,355.678600
2100,317.706100
2400,312.796200
2700,304.668300
3000,303.974800


TrainOutput(global_step=16780, training_loss=261.3095660058477, metrics={'train_runtime': 19289.1368, 'train_samples_per_second': 27.834, 'train_steps_per_second': 0.87, 'total_flos': 6.204301306785627e+19, 'train_loss': 261.3095660058477, 'epoch': 10.0})

### Load the trained model checkpoint and evaluate on the Dhivehi test set (greedy decoding)

In [35]:
from transformers import AutoModelForCTC, Wav2Vec2BertProcessor
import torch
import numpy as np

all_pred = []
all_refs = []

model.eval()

checkpoint_dir = "/content/kenlm/build/outputs/checkpoint-16780"

model = AutoModelForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2BertProcessor.from_pretrained(checkpoint_dir)

for ex in dv_test_p:
    input_feats = torch.tensor(ex["input_features"]).unsqueeze(0).to(model.device)

    with torch.no_grad():
        logits = model(input_feats).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.batch_decode(pred_ids)[0]

    ref_text = processor.decode(ex["labels"], group_tokens=False)

    # strip language token (<dv>) before scoring
    pred_text = pred_text.replace("<dv>", "").strip()
    ref_text  = ref_text.replace("<dv>", "").strip().lower()

    all_pred.append(pred_text)
    all_refs.append(ref_text)

wer = wer_metric.compute(predictions=all_pred, references=all_refs)
cer = cer_metric.compute(predictions=all_pred, references=all_refs)

print(f"WER (Dhivehi only, greedy): {wer:.4f}")
print(f"CER (Dhivehi only, greedy): {cer:.4f}")

for idx in range(min(3, len(all_refs))):
    print("\n--- Example", idx, "---")
    print("REF :", all_refs[idx])
    print("PRED:", all_pred[idx])


WER (Dhivehi only, greedy): 0.4302
CER (Dhivehi only, greedy): 0.0660

--- Example 0 ---
REF : ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން
PRED: ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން

--- Example 1 ---
REF : އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާ ވެސް އިތުރު ސުވާލެއް ނުކުރޭ
PRED: އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާވެސް އިތުރު ސުވާލެއް ނުކުރޭ

--- Example 2 ---
REF : އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ
PRED: އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ
